<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica%20Tema%2013.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica Tema 13

**Clase:** Fundamentos Algoritmos de Aprendizaje Automatico

**Tema:** Serializacion de Modelos y Conexion con API

## reto 1. entrenamiento y serializacion del pipeline completo con joblib

aquí lo que hice fue entrenar el modelo y después guardarlo con Joblib. La idea es que no tenga que volver a entrenar el modelo cada vez que lo quiera usar. Básicamente guardo todo el pipeline completo, incluyendo el escalado y la regresión logística, y después verifico que sí se pueda cargar otra vez correctamente.

In [1]:
from sklearn.pipeline import Pipeline as pipeline
from sklearn.preprocessing import StandardScaler as standard_scaler
from sklearn.linear_model import LogisticRegression as logistic_regression
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from joblib import dump, load
from pathlib import Path
import json
import sklearn
import joblib

x, y = make_classification(n_samples=1000, n_features=4, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

pipe_churn = pipeline([
    ("scaler", standard_scaler()),
    ("clf", logistic_regression(max_iter=1000))
])

pipe_churn.fit(x_train, y_train)
model_path = Path("modelos/v1")
model_path.mkdir(parents=True, exist_ok=True)
dump(pipe_churn, model_path / "model.joblib")

loaded_model = load(model_path / "model.joblib")
loaded_predictions = loaded_model.predict(x_test)

print("model serialized successfully")
print("file exists:", (model_path / "model.joblib").exists())
print("test accuracy:", accuracy_score(y_test, loaded_predictions))


model serialized successfully
file exists: True
test accuracy: 0.885


## reto 2. registro de metadatos de entrenamiento

aquí agregué los metadatos del modelo. Guardé cosas como el nombre, la versión, el accuracy y las librerías utilizadas. Esto sirve porque si después tengo varias versiones del modelo, puedo saber exactamente cuál estoy usando, cómo rindió y con qué entorno fue creado.

In [2]:
offline_accuracy = accuracy_score(y_test, loaded_predictions)

metadata = {
    "model_name": "churn_logistic_regression",
    "version": "v1",
    "offline_accuracy": float(offline_accuracy),
    "libraries": {
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__
    }
}

with open(model_path / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print(metadata)


{'model_name': 'churn_logistic_regression', 'version': 'v1', 'offline_accuracy': 0.885, 'libraries': {'scikit_learn': '1.6.1', 'joblib': '1.6.0'}}


Los metadatos conservan el nombre, version, rendimiento y librerias del modelo. Esto ayuda a reproducir el experimento y evita perder contexto cuando el modelo se lleva a produccion.

## reto 3. definicion del contrato de entrada con pydantic

aqui se define con Pydantic cómo deben llegar los datos a la API. Como el modelo espera cuatro variables numéricas, el esquema obliga a que esas cuatro entradas sean floats. Esto evita que al modelo le lleguen datos mal formados o con tipos incorrectos.

In [3]:
from pydantic import BaseModel as base_model

class prediction_input(base_model):
    feature_1: float
    feature_2: float
    feature_3: float
    feature_4: float

example_input = prediction_input(
    feature_1=0.5,
    feature_2=-1.2,
    feature_3=2.0,
    feature_4=0.8
)

print(example_input)


feature_1=0.5 feature_2=-1.2 feature_3=2.0 feature_4=0.8


Pydantic valida el formato antes de hacer inferencia. Asi se reducen errores por tipos incorrectos o datos mal estructurados.

## reto 4. implementacion del endpoint de inferencia y umbral operativo

aqui se crea la estructura del endpoint con FastAPI. El modelo se carga una sola vez y cuando llega una petición se obtiene la probabilidad de la clase positiva. Después uso un threshold de 0.5 para decidir si la salida es clase 0 o clase 1.

In [4]:
from fastapi import FastAPI as fast_api
import numpy as np

app = fast_api()
service_model = load("modelos/v1/model.joblib")
threshold = 0.5

@app.post("/predict")
def predict(data: prediction_input):
    features = np.array([[
        data.feature_1,
        data.feature_2,
        data.feature_3,
        data.feature_4
    ]])

    probability = service_model.predict_proba(features)[0, 1]
    predicted_class = int(probability >= threshold)

    return {
        "probability": float(probability),
        "predicted_class": predicted_class,
        "threshold": threshold
    }


Devolver la probabilidad da mas flexibilidad que devolver solo 0 o 1, porque permite cambiar el umbral de decision segun las necesidades del sistema sin volver a entrenar el modelo.

## conclusion

La practica muestra como pasar de un modelo entrenado a una estructura preparada para produccion: guardar el pipeline, registrar metadatos, validar entradas y exponer predicciones mediante una API.